# 02 - LLaMEA Evolutionary Synthesis Pipeline

Multi-process parallel algorithm discovery across the BBOB problem benchmark matrix with SQLite persistence and auto-resumption.


In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import numpy as np

# Ensure project root & src are in path
cwd = Path(".").resolve()
root_dir = cwd.parent if cwd.name == "notebooks" else cwd
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from shared.config import CONFIGS_DIR, DATA_DIR, RESULTS_DIR
from shared.database import initialize_sqlite_storage
from evolution.infra.storage.campaigns.repository import ExperimentConfigRepository
from evolution.application.experiment_service import EvolutionExperimentService
from evolution.infra.llm.client import LLMClient
from evolution.domain.enums import PromptStrategy

print("[OK] LLaMEA Evolution Pipeline initialized.")

## 1. LLM Provider Connection
Initialize and verify the LLM connection configured in your `.env` file.


In [ ]:
env_path = root_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)

llm_provider = os.getenv("LLM_PROVIDER", "local")
try:
    llm = LLMClient(llm_provider)
    print("=" * 65)
    print(f"Initialized Provider: {llm_provider}")
    print(f"Model Target:         {getattr(llm, 'model', 'unknown')}")
    print("=" * 65)
except Exception as e:
    print(f"[!] Warning: Could not connect to LLM server ({e}).")
    print("    Initializing LLMClient with skip_validation=True for offline inspection.")
    llm = LLMClient(llm_provider, skip_validation=True)
    print("=" * 65)
    print(f"Initialized Provider (Offline): {llm_provider}")
    print(f"Model Target:                   {getattr(llm, 'model', 'unknown')}")
    print("=" * 65)


## 2. Load Configuration & Experiment Matrix (`experiments.toml` & `runner.toml`)
Reads the target problems, dimensions, prompt strategies, noise levels from `experiments.toml`, execution and retry knobs from `runner.toml`, and builds the visual execution matrix.


In [ ]:
# 1. Initialize Repositories and Evolution Application Service
sqlite_repo = initialize_sqlite_storage()
config_repo = ExperimentConfigRepository()
evolution_service = EvolutionExperimentService(
    sqlite_repo=sqlite_repo,
    config_repo=config_repo,
    llm_client=llm,
)

# 2. Audit Matrix Status against SQLite Database for active LLM model
matrix_df, summary = evolution_service.audit_campaign()

print("=" * 80)
print(f"🎯 Active Matrix Configuration for \"{summary['model_name']}\":")
print(f"   • Filtered Matrix Size:    {summary['total_conditions']} conditions")
print(f"   • Valid Completed:         {summary['completed_conditions']}/{summary['total_conditions']} ({summary['progress_pct']:.1f}%)")
print(f"   • Failed to Retry:         {summary['retry_conditions']}")
print(f"   • Retry Failed Synthesis:  {summary['retry_failed_synthesis']}")
print(f"   • Auto-Resume Running:     {summary['auto_resume']}")
print(f"   • Skip Completed:          {summary['skip_completed']}")
if summary['filter_problems']:   print(f"   • Filter Problems:         f{summary['filter_problems']}")
if summary['filter_dims']:       print(f"   • Filter Dimensions:       {summary['filter_dims']}D")
if summary['filter_modes']:      print(f"   • Filter Modes:            {summary['filter_modes']}")
if summary['filter_strategies']: print(f"   • Filter Strategies:       {summary['filter_strategies']}")
if summary['target_exp_ids']:    print(f"   • Target Experiment IDs:   {summary['target_exp_ids']}")
print("=" * 80)
display(matrix_df) if "display" in globals() else print(matrix_df.to_string(index=False))

## 3. Parallel Multi-Process Experiment Execution (With Auto-Resumption & DB Sync)
Automatically queries `data/db.sqlite3` to check which experiments are still pending or incomplete:
1. **Interrupted Runs (`status == 'running'`)**: Fetched from the DB and resumed directly from their last completed iteration without needing re-configuration.
2. **Completed Runs (`status == 'completed'`)**: Automatically skipped if they already produced a valid champion algorithm.
3. **Failed Synthesis Runs (`best_final_error is null`)**: Automatically retried if `retry_failed_synthesis = true`.
4. **Pending Runs**: Created as fresh tasks for the remaining required runs.

All tasks execute concurrently across worker processes using `TaskOrchestrator` (`ProcessPoolExecutor`) with safe SQLite WAL concurrency.

In [ ]:
# 1. Build the task list based on execution and filter settings
tasks = evolution_service.build_tasks()
cfg = config_repo.load_config()

print("=" * 80)
print(f"📊 Experiment Dispatch Summary for '{llm.model.name}':")
print(f"   • Total Tasks to Execute:       {len(tasks)}")
print(f"   • Parallel Worker Processes:    {cfg['num_processes']}")
print("=" * 80)

# 2. Execute tasks in parallel using TaskOrchestrator encapsulated in EvolutionExperimentService
if len(tasks) == 0:
    print(f"🎉 All requested experiments are already completed with valid champions for '{llm.model.name}'! Nothing to run.")
    results = {}
else:
    print(f"🚀 Dispatching {len(tasks)} tasks across worker processes...")
    results = evolution_service.run_campaign()